In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import plotly
import math
import shap
from catboost import CatBoostRegressor
import matplotlib.pyplot as plt
from typing import List
import plotly.graph_objs as go
from plotly.subplots import make_subplots
from data_profiling import ProfileReport
from plotly.subplots import make_subplots
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from IPython.display import display
plotly.io.renderers.default = "notebook"
%matplotlib inline

# Table of Contents
1. [Feature Descriptions and Profiling](#Feature-Descriptions-and-Profiling)
2. [Time Series Vizual Analysis](#Time-Series-Vizual-Analysis)
3. [Feature Dependency Analysis](#Feature-Dependency-Analysis)
4. [Data Cleaning and Grouping](#Data-Cleaning-and-Grouping)
5. [What we achieved and what is the problem](#What-we-achieved-and-what-is-the-problem)
6. [Mahalanobis distance study to identify anomaly region](#Mahalanobis-distance-study-to-identify-anomaly-region)
7. [Noise study](#Noise-study)
8. [Conclusions](#Conclusions)

# Feature Descriptions and Profiling

In [ ]:
df = pd.read_parquet('../data/01_raw/df_train_test.parquet')

In [ ]:
df.shape

In [ ]:
df.head(3)

The timestamps are taken every 10 mins.

We see that the df index is not timestamps. For time series analysis, it's better to have it as a timestamp, so let's make it.

This is especially important when plotting time series and understanding different phenomena over time.

In [ ]:
df.index = pd.to_datetime(df['Timestamps'])
df.drop(columns='Timestamps', inplace=True)

Before doing any analysis, it's critical to understand what each raw feature means. So, let's describe it.

In [ ]:
df.columns

| **Feature** | **Description** | **Typical Units** | **Category** |
|--------------|----------------|-------------------|---------------|
| **WindSpeed** | Mean wind speed measured over the recording interval by the anemometer on the nacelle. | m/s | Meteorological | 
| **WindDirAbs** | Absolute wind direction — measured with respect to geographic North. | ° (degrees) | Meteorological |
| **WindDirRel** | Relative wind direction — difference between wind direction and nacelle yaw position (turbine facing direction). | ° | 
| **Power** | Electrical power output delivered by the generator during the sampling interval (average). | kW | Performance |
| **Pitch** | Average blade pitch angle (rotation of blades around their longitudinal axis to control aerodynamic load). | ° | Control |
| **GenRPM** | Generator rotational speed (after gearbox). | rpm | Mechanical |
| **RotorRPM** | Rotor rotational speed (before gearbox). | rpm | Mechanical |
| **EnvirTemp** | Ambient environmental temperature near the nacelle. | °C | Environmental |
| **NacelTemp** | Temperature measured inside the nacelle (housing on top of the tower). | °C | Environmental |
| **GearOilTemp** | Temperature of gearbox lubricating oil (indicator of mechanical load and thermal stress). | °C | Mechanical / Condition
| **GearBearTemp** | Temperature of the main gearbox bearing. | °C | Mechanical / Condition Monitoring |
| **GenPh1Temp** | Temperature of generator winding Phase 1. | °C | Electrical / Condition Monitoring |
| **GenBearTemp** | Temperature of generator bearing (critical indicator of bearing wear or lubrication issues). | °C | Condition Monitoring |

## Let's make a quick dataset description

In [ ]:
df.describe()

**We see that:**
- There are weird negative values for min_values of some features (e.g. WindSpeed, GenRPM) which don't make physical sense. Maybe our data is quite noisy or have outlying values we need to filter.
- Some parameters have a high **std / mean value**, so we have high variability within the data.

However, basic description does not give much info.

Let's use the Data Profiler.

In [ ]:
profile = ProfileReport(df)
profile

**Some observations from parameter values and distributions**
- There are outliers including negative values for many parameters, definitely needs to be cleaned out.
- A lot of the values of are zero (close to 10%), which means that a lot of time the turbine does don't work. It also means that these regime needs to be most likely cleaned out when analyzing the relatinsionships, correlations, etc. It can also be the turbine downtime.
- While many parameter distributions are close to normal (except zero values), **Power** - a very important parameter, is highly skewed If we gonna use it as a model target, it can be a problem (expecially in linear regression)
- GenRPM is relatuvely evenly distributed but also has some certain peaks which seem to be the main operating regimes.
- The temperature seem to be moderate and does not have much of negative values as well as high values which could indicate thermo-intensive regimes.

# Time Series Vizual Analysis

In [ ]:
def plot_time_series(df: pd.DataFrame, columns: List[str], step=10, rolling_window=None):
    """
    Clean and fast Plotly plot:
    - Subplots stacked vertically
    - Optional rolling median trend (black)
    - Only shows every `step`th tick

    Args:
        df (pd.DataFrame): Time series dataframe with a DateTimeIndex.
        columns (List[str]): List of column names to plot.
        step (int, optional): Subsampling step for faster plotting. Defaults to 10.
        rolling_window (_type_, optional):  Window size for rolling median. Defaults to None. If None → no rolling median plotted.
    """
    
    # subsample (for speed)
    df_small = df.iloc[::step]
    
    # subplot layout
    fig = make_subplots(
        rows=len(columns),
        cols=1,
        shared_xaxes=True,
        subplot_titles=columns
    )
    
    # create rolling dataframe, if needed
    if rolling_window is not None:
        df_rolling=(
            df[columns]
            .rolling(window=rolling_window, min_periods=1)
            .median()
            .iloc[::step]
        )
    
    for i, col in enumerate(columns, start=1):
        # original series 
        fig.add_trace(
            go.Scatter(
                x=df_small.index,
                y=df_small[col],
                mode='lines',
                name=col,
                line=dict(width=1)
            ),
            row=i, col=1
        )
        # rolling series, only if needed
        if rolling_window is not None:
            fig.add_trace(
                go.Scatter(
                    x=df_rolling.index,
                    y=df_rolling[col],
                    mode='lines',
                    name=f'{col} (rolling median)',
                    line=dict(width=1, color='black')
                ),
                row=i, col=1
            )
        
        
    fig.update_xaxes(tickmode="auto")

    fig.update_layout(
        height=250 * len(columns),
        showlegend=False,
        title_text="Time Series Overview",
        margin=dict(l=50, r=30, t=50, b=50)
    )
        
    fig.show()
    

In [ ]:
df.columns

In [ ]:
cols_to_plot = [
    'Power', 'WindSpeed', 'GenRPM', 'RotorRPM', 
    'WindDirAbs', 'WindDirRel', 'Pitch',
    'EnvirTemp', 'NacelTemp', 'GearOilTemp',
    'GearBearTemp', 'GenPh1Temp', 'GenBearTemp'
]
plot_time_series(df, cols_to_plot, step=1)

Spikes may be outliers: 
- if you don’t want to predict outliers, exclude the outliers
- If you want to model anomalies as a point, keep them

In our case, we are not looking at anomaly as a point

0 chunk in the data is the turbine shutdown that we want to predict, we want to predict before that

If the wind is unstable, the power will be unstable

If the behaviour is unsteady, building features (stat or domain-based), it is important to take the behaviour in consideration

Lot of noise (GearBearTemp) - expected in physical operators - the noise is what makes your model worse

Seasonality (EnvirTemp)

What can i see before the downtime - is there an identifiable signal - no

It’s good to plot rolling median plots on such data/ features

Black - rolled/ smoothed value - smoothing (mean/ median/ gaussian/ exponential)

Helpful to see trend (eg envtemp)

Median filter is great for outlier removal (GenBearTemp) - works for vast majority cases

We can see the power follows some trends of windspeed, genrm, rotorrpm etc



- We see that most of the parameters have strong outlying values. This can be a problem when fitting the model.
- We see that there is a data chunk where all the values are zero. This corresponds to the turbine shoutdown.
- The operation of the turbine is unsteady which is expeted because the wind has turbulent and intermittent nature.
- Many signals have quite A LOT of noise, at least visually. It might be a good idea to analyze it and denoise if possible.
- Some signals have seasonality, especially tempearture-reated which makes sense.
- We don't observe any strong weird anomaly behavior any time before the shoutdown. 

Let's also plot the time series with median rolling that helps us see the trends.

In [ ]:
cols_to_plot = [
    'Power', 'WindSpeed', 'GenRPM', 'RotorRPM', 
    'WindDirAbs', 'WindDirRel', 'Pitch',
    'EnvirTemp', 'NacelTemp', 'GearOilTemp',
    'GearBearTemp', 'GenPh1Temp', 'GenBearTemp'
]
plot_time_series(df, cols_to_plot, step=5, rolling_window=15)

- Here, if zooming in, it's possible to see that there is a lot of noise in parameters that can be smoothed out.
- Clearly,  in addition to noise, even the median parameter values have high variability.
- We can also see that we can use the median filter as the outlier removal.
- The smoothed values show better relationships betwen Power, Wind Speed, GenRMP and RotorRPM parameters, especially when there are big ups and downs. These parameters might have good feature <--> target relationships.
- The same we can see for GenPh1Temp and GenBearTemp, NacelTemp and EnvirTemp.

# Feature Dependency Analysis

In [ ]:
# compute correlation matrix of features
corr = df.corr()

# plot correlation heatmap
plt.figure(figsize=(15, 15))
sns.heatmap(corr, annot=True, cmap="coolwarm", center=0, fmt=".2f", square=True, annot_kws={"size": 12})
plt.title("Feature Correlation Heatmap")

Heatmp of parameters

Shouldn’t be >0.95 - strong multicollonearity - bad


- We see that some parameters have VERY good correlations like Power <--> GenRPM corr=0.88
- Some parameters are barely correlated with anything, e.g. Pitch.
- There are many moderate to strong correlation values.
- What is weird is that when we analyze timeseries, we did NOT see such strong relationships, especially taking into account the noise.

Let's take Power as an example and sort correlations from highest to lowest.

In [ ]:
# Compute correlation of all columns with the target
corr = df.corr()['Power'].drop('Power')

# Sort by absolute correlation value
correlations_sorted = corr.reindex(corr.abs().sort_values(ascending=False).index)
correlations_sorted

From the Time Series plots, we haven't seen such a strong correlation. Let's check the scatter-like plots.

In [ ]:
def plot_relationships(x: pd.Series, y: pd.Series, x_label: str='X', y_label: str='Y'):
    """
        Plots x-y relationships in different formats (Regression, KDE and HexBin plots)
    """
    fig, axs = plt.subplots(1, 3, figsize=(18,5))
    
    # regression plot
    sns.regplot(x=x, y=y, ax=axs[0], scatter_kws={'s': 20}, line_kws={'color': 'red'})
    axs[0].set_title('Regression plot')
    axs[0].set_xlabel(x_label)
    axs[0].set_ylabel(y_label)
    
    # kde plot
    sns.kdeplot(x=x, y=y, fill=True, cmap="mako", ax=axs[1], thresh=0.01)
    axs[1].set_title('KDE plot')
    axs[1].set_xlabel(x_label)
    axs[1].set_ylabel(y_label)
    
    # hexplot
    plt.hexbin(x, y, gridsize=30, cmap='viridis', mincnt=1)
    axs[2].set_title('Hexbin plot')
    axs[2].set_xlabel(x_label)
    axs[2].set_ylabel(y_label)
    
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()

In [ ]:
v1 = 'GenRPM'
v2 = 'Power'
n = 10
df_local = df.copy() # df[(df[v1] > 800)]
plot_relationships(df_local[v1][::n], df_local[v2][::n], v1, v2)

Regression, KDE, Hexbin plot of GenRPM  to Power:
    
the relationship is far from correlated

In the density plot, completely different probability mass around 0 (missing values 10%) highly influences the correlation value


These variables do NOT look like well-correlated.

The correlation value is strongly effected by the outliers. But in this case making it high!

For the outliers, we can see that there are some clouds of points which are separated from the main relationship.

There is a big cloud of points around zero which drives big correlation.

Let's check for some more variables.

In [ ]:
v1 = 'WindSpeed'
v2 = 'Power'
n = 10
df_local = df.copy()
plot_relationships(df_local[v1][::n], df_local[v2][::n], v1, v2)

We see the same picture here. We also see that there are MANY points that looks like one point in zero. 

This is NOT possible to see in the scatter plot, but we can see it in the KDE and Hexbin plots.

We clearly see that the correlation values are strongly influecned by the outliers, especially zeros.

Let's first remove zeros.

In [ ]:
df_no_zero = df[df['Power'] > 20]

In [ ]:
# Compute correlation of all columns with the target
corr = df_no_zero.corr()['Power'].drop('Power')

# Sort by absolute correlation value
correlations_sorted = corr.reindex(corr.abs().sort_values(ascending=False).index)
correlations_sorted

Previous correlation values:

In [ ]:
# GenRPM          0.879374
# GenPh1Temp      0.828008
# GearOilTemp     0.743265
# WindSpeed       0.705276
# RotorRPM        0.703314
# GearBearTemp    0.703189
# GenBearTemp     0.677292
# WindDirAbs      0.419142
# NacelTemp       0.313068
# EnvirTemp       0.249844
# Pitch           0.104456
# WindDirRel     -0.015779

Now we see much smaller correlations.

Let's check how it looks in scatters.

In [ ]:
v1 = 'GenRPM'
v2 = 'Power'
n = 10
df_local = df_no_zero.copy()
plot_relationships(df_local[v1][::n], df_local[v2][::n], v1, v2)

In [ ]:
v1 = 'RotorRPM'
v2 = 'Power'
n = 10
df_local = df_no_zero.copy()
plot_relationships(df_local[v1][::n], df_local[v2][::n], v1, v2)

We still see the influence by the outlers. 
    
And now it's hard to say if the outliers increase or decrease correlations.

But what is more important, it's hard to see the relationships clearly and trully understand the data.

Let's check the distributions more closely.

In [ ]:
# # plot the distribution of each feature

n_cols = 3
n_rows = int(np.ceil(len(df.columns) / n_cols))

plt.figure(figsize=(15,10))

for i, col in enumerate(df.columns, 1):
    plt.subplot(n_rows, n_cols, i)
    sns.histplot(df_no_zero[col].dropna(), bins=50, kde=True)
    plt.title(col, fontsize=10)
    plt.xlabel('')
    plt.ylabel('')
    
plt.tight_layout()

We can see long tails in most of the features, but the number of data points is not that big.

Let's also check if we can see in the multidimentional space reduced to 2 dimentions with PCA.

Note: A combination of features can also be an anomaly

Projects every point into the pc axes, transform into 2D, then do clustering

Before pca, it is good practice to apply standard scaling

Check explained variance and cumulative variance plot (how much info you lose when projecting to smaller dimension)

Reduce to 2d and plot (df vs df_no_zero): most outliers are 0 rows

2 components represent: 62%, so not totally reliant, but still useful


In [ ]:
# Standardize the features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df) # df_no_zero

In [ ]:
# Apply PCA
pca = PCA(n_components=len(df.columns))
X_pca = pca.fit_transform(X_scaled)

# Create DataFrame of first 2 components
pca_df = pd.DataFrame(data=X_pca[:, :2], columns=["PC1", "PC2"])

In [ ]:
# Plot 1: PCA Scatter Plot (2D)
plt.figure(figsize=(8, 5))
sns.scatterplot(x="PC1", y="PC2", data=pca_df[::1])
plt.title("PCA: First 2 Principal Components")
plt.grid(True)
plt.tight_layout()
plt.show()

We can see that there are outlying values.

If we check this for **df_no_zero**, we will not see them.

So, this means that these PCA outlying values are zeros.

Also, let's check how representative the first two principal components are.

In [ ]:
# Plot 2: Explained Variance (Scree Plot)
explained_variance = pca.explained_variance_ratio_
cumulative_variance = explained_variance.cumsum()
cumulative_variance

In [ ]:
explained_variance

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(range(1, len(cumulative_variance) + 1), cumulative_variance, marker='o', linestyle='-')
plt.xlabel("Number of Principal Components")
plt.ylabel("Cumulative Explained Variance")
plt.title("Cumulative Explained Variance")
plt.grid(True)
plt.tight_layout()
plt.show()

Also, we see that a lot of information missing, so overall, PCA plots might not be very representative.

It allowed us to detect only the most obvious "outliers" (which are zeros) which we cleaned easily anyway.

This again emphasize the noisy nature of the data. In the multidimentional space, even the obvious outliers are hard to detect.

In this case, the best way to clean them is feature by feature.

Let's do this.

# Data Cleaning and Grouping

### Z-Score Filter

Z-score tells you how many standard deviations a value is from the mean

Usually, if the parameter value z-score >3 (but it can be 4 or 5), it can be considered an anomaly

Let's try to remove the outliers with a simple Z-score filter

In [ ]:
def remove_outliers_zscore(df:pd.DataFrame, threshold:float=3, nan_treatment:str='ffill'):
    """
    Replace outliers (based on z-score) with NaN for each numeric column
    and report how many values were replaced.

    Parameters
    ----------
    df : pd.DataFrame
        Input DataFrame (numeric or mixed).
    threshold : float, optional
        Z-score threshold. Default = 3.
    nan_treatment: str
        The way we treat nans (ffill or drop)
    Returns
    -------
    df_masked : pd.DataFrame
        DataFrame with outlier values replaced by NaN.
    """
    df_masked = df.copy()
    numeric_cols = df.select_dtypes(include=np.number).columns

    total_replaced = 0
    replaced_per_column = {}

    for col in numeric_cols:
        mean = df[col].mean()
        std = df[col].std(ddof=0)
        z_score = np.abs((df[col] - mean) / std)

        outlier_mask = z_score > threshold
        n_replaced = outlier_mask.sum()

        df_masked.loc[outlier_mask, col] = np.nan

        replaced_per_column[col] = n_replaced
        total_replaced += n_replaced

    # Treat outliers
    if nan_treatment == 'ffill':
        df_masked = df_masked.ffill()
    elif nan_treatment =='drop':
        df_masked = df_masked.dropna()
    else:
        raise ValueError(f'{nan_treatment} nan_treatment is not recognized')

    print(f"Replaced {total_replaced} values total (|z| > {threshold}).")
    print("Per column replacements:")
    for col, n in replaced_per_column.items():
        print(f"  {col}: {n}")

    return df_masked

In [ ]:
df_clean = remove_outliers_zscore(df_no_zero, nan_treatment='ffill', threshold=3) # Check with 2-3

In [ ]:
# Select numeric columns only
n_cols = 3
n_rows = int(np.ceil(len(df.columns) / n_cols))

plt.figure(figsize=(15, 10))

for i, col in enumerate(df.columns, 1):
    plt.subplot(n_rows, n_cols, i)
    sns.histplot(df_clean[col].dropna(), bins=25, kde=True)
    plt.title(col, fontsize=10)
    plt.xlabel('')
    plt.ylabel('')

plt.tight_layout()

We see that we cut the outliers quite well. Now, let's check the X-Y plots.

In [ ]:
v1 = 'GenRPM'
v2 = 'Power'
n = 10
df_local = df_clean.copy()
plot_relationships(df_local[v1][::n], df_local[v2][::n], v1, v2)

In [ ]:
v1 = 'WindSpeed'
v2 = 'Power'
n = 10
df_local = df_clean.copy()
plot_relationships(df_local[v1][::n], df_local[v2][::n], v1, v2)

Z-score removes the long tails in distribution, and outliers in regression plot_time_series

The relationship is clearer in regression plot. There is still some noise. So let's aggregate the data over 6h

Now, we can better see the relationships. 

However, due to the noisy nature of the signals, it's still hard.

Let's try one trick - let's plot grouped data.

## Grouped data plots

In [ ]:
df_gr = df_clean.resample('6h').mean()

In [ ]:
v1 = 'GenRPM'
v2 = 'Power'
n = 1
df_local = df_gr.copy()
plot_relationships(df_local[v1][::n], df_local[v2][::n], v1, v2)

In [ ]:
v1 = 'WindSpeed'
v2 = 'Power'
n = 1
df_local = df_gr.copy()
plot_relationships(df_local[v1][::n], df_local[v2][::n], v1, v2)

In [ ]:
v2 = "Power"         # fixed y-variable
df_local = df_gr.copy()
n = 1                # subsampling step

for col in df_local.columns:
    if col == v2:
        continue     # skip Power itself
    print(f"Plotting: {col} vs {v2}")
    plot_relationships(
        df_local[col][::n],
        df_local[v2][::n],
        col,
        v2
    )

Now we can see that ON AVERAGE for some variables like WindSpeed or GenRPM vs Power the relationships are non-linear, but there is some relationship, which can be picked up gradient boosting/ neural networks

We can take this into account because maybe it can be useful when creating the model and identifying the time horizon for the model prediction.

Some varibleas are close to linear dependency, but the variance is very high, e.g. EnvirTemp vs Power.

Pitch has a very strange relationship with Power, however, in some data ranges it can be a useful predictor.

Let's see how it looks as a time series.

When we stride without resampling the relationship is clearer (aggregation and pooling effect without missing data)

In [ ]:
cols_to_plot = [
    'Power', 'WindSpeed', 'GenRPM', 'RotorRPM', 
    'WindDirAbs', 'WindDirRel', 'Pitch',
    'EnvirTemp', 'NacelTemp', 'GearOilTemp',
    'GearBearTemp', 'GenPh1Temp', 'GenBearTemp'
]

plot_time_series(df_gr, cols_to_plot, step=1)

We can see and study the time series more clearly after grouping.

We can see how big ups and downs for high correlation features match ups and downs of Power.

However, we can't really see any specific behavior before the the downtime.

Let's try to do something simple - apply PCA to the grouped data.

The idea is that if there is a region in a multidimentional space that is abnormal to the main cloud of points, this might correspond to the anomaly behavior that we want to see.

We saw that on the original data it did not work but it can be because of the high noise in the data.

### PCA on Grouped Data

In [ ]:
df_pca = df_gr.copy().dropna()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_pca)

pca = PCA(n_components=len(df_pca.columns))
X_pca = pca.fit_transform(X_scaled)


pca_df = pd.DataFrame(
    data=X_pca[:, :2],
    index=df_pca.index,
    columns=["PC1", "PC2"]
)

In [ ]:
# Plot 1: PCA Scatter Plot (2D)
plt.figure(figsize=(8, 5))
sns.scatterplot(x="PC1", y="PC2", data=pca_df[::1])
plt.title("PCA: First 2 Principal Components")
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# Plot 2: Explained Variance (Scree Plot)
explained_variance = pca.explained_variance_ratio_
cumulative_variance = explained_variance.cumsum()
cumulative_variance

- We see that even after grouping, first 2 PC still do not explain eniough of variance.
- We can see some "cluster" of points for PC_1 > 4, but it's not very well separated.
- However, let's plot it on time series.

In [ ]:
pc1_thr = 4
pc1 = pca_df["PC1"]

mask_red = pc1 > pc1_thr   # positions where PC1 is "high"

cols_to_plot = [
    'Power', 'WindSpeed', 'GenRPM', 'RotorRPM', 
    'WindDirAbs', 'WindDirRel', 'Pitch',
    'EnvirTemp', 'NacelTemp', 'GearOilTemp',
    'GearBearTemp', 'GenPh1Temp', 'GenBearTemp'
]

for col in cols_to_plot:
    plt.figure(figsize=(10, 3))

    # base time series (grey line)
    plt.plot(
        df_pca.index,
        df_pca[col],
        color="grey",
        linewidth=1,
        label=col
    )

    # mark PC1 > threshold in red
    plt.scatter(
        df_pca.index[mask_red],
        df_pca.loc[mask_red, col],
        color="red",
        s=10,
        label=f"PC1 > {pc1_thr}"
    )

    plt.title(f"{col} – points where PC1 > {pc1_thr} in red")
    plt.xlabel("Time")
    plt.ylabel(col)
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.show()

- We see that PCA "outliers" are just the points with highest values of Power, WindSpeed, GenRPM, etc.
- We see that these values happened far before the downtime and quite quickly after downtime.
- So, these points can hardly be considered as anomalies in terms of downtime detection.

# What we achieved and what is the problem

- We understand that the data has a lot of noise that can be averaged out.
- We understand the main relationships between the parameters.
- We can clean only basic "outlier / anomaly" regime - when the tuurbine is stopped.
- We do not see strong anomaly behavior in 1D (per feature) close to downtime.
- We do not see any anomaly identified via PCA, both on averaged and on raw data.
- **We still don't know if (and how) we can identify the anomaly behavior consistenly**